## N-BEATS v3 — Friend recipe (Kaggle GPU)

**Adopted from teammate notes:**
- Preprocess: clip≥0 · weekly grid gap-fill(0) · **log1p** · **per-series Min–Max** (train-only)
- Univariate Store×Dept sales only
- Validation: **3 rolling-origin folds** (Holiday / Spring / Late) → mean WMAE
- Search: **Optuna** (~25 trials): stacks, blocks, layers, width, dropout, lookback, horizon, batch, lr
- Loss: L1 · 25 epochs · **max 10 windows / series**

**Run on Kaggle:** Accelerator **GPU T4** · Internet On · Attach competition data · **Save & Run All**

Also works on Colab. `FAST_RUN=True` → 5 trials smoke test.


In [ ]:
#1 — env
import os
from pathlib import Path
for root in ['/kaggle/input', '/content/kaggle/input']:
    if Path(root).exists():
        print('Found', root); break
else:
    print('No data yet')


In [ ]:
#2 — setup
NOTEBOOK_VERSION = 'nbeats_v3_friend_optuna_kaggle'
!pip install -q dagshub mlflow wandb kaggle optuna

import json, os, time, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import optuna
from optuna.samplers import TPESampler
import mlflow, dagshub

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

if Path('/kaggle/working').exists():
    WORK_DIR = Path('/kaggle/working')
elif Path('/content').exists():
    WORK_DIR = Path('/content')
else:
    WORK_DIR = Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

PROGRESS_LOG = WORK_DIR / 'nbeats_progress.log'
RUN_COMPLETE = WORK_DIR / 'RUN_COMPLETE.txt'
STUDY_CSV = WORK_DIR / 'nbeats_optuna_trials.csv'
for p in [RUN_COMPLETE, PROGRESS_LOG]:
    if p.exists(): p.unlink()

def log_progress(msg):
    line = f"[{datetime.utcnow().isoformat()}Z] {msg}"
    print(line)
    with open(PROGRESS_LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

def pick_device():
    if torch.cuda.is_available():
        try:
            _ = nn.Linear(8, 4).cuda()(torch.randn(4, 8, device='cuda'))
            torch.cuda.synchronize()
            log_progress(f'CUDA OK: {torch.cuda.get_device_name(0)}')
            return torch.device('cuda'), True
        except Exception as e:
            log_progress(f'CUDA fail ({e})')
    log_progress('CPU'); return torch.device('cpu'), False

device, USE_CUDA = pick_device()
if USE_CUDA: torch.cuda.manual_seed_all(SEED)

tok = get_secret('DAGSHUB_USER_TOKEN')
if tok: dagshub.auth.add_app_token(tok)
dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
mlflow.set_experiment('NBEATS_Training')
log_progress(f'{NOTEBOOK_VERSION} | {WORK_DIR}')


In [ ]:
#3 — load data
COMPETITION_SLUG = 'walmart-recruiting-store-sales-forecasting'

def read_competition_csv(data_dir, stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(stem)

def find_data_dir():
    for c in [f'/kaggle/input/competitions/{COMPETITION_SLUG}', f'/kaggle/input/{COMPETITION_SLUG}',
              f'/content/kaggle/input/competitions/{COMPETITION_SLUG}']:
        if os.path.isdir(c) and (os.path.exists(f'{c}/train.csv') or os.path.exists(f'{c}/train.csv.zip')):
            return c
    return f'/content/kaggle/input/competitions/{COMPETITION_SLUG}'

DATA_DIR = find_data_dir()
if not (os.path.exists(f'{DATA_DIR}/train.csv') or os.path.exists(f'{DATA_DIR}/train.csv.zip')):
    token = get_secret('KAGGLE_API_TOKEN')
    if not token: raise RuntimeError('Need KAGGLE_API_TOKEN or attach Kaggle input')
    os.environ['KAGGLE_API_TOKEN'] = token
    kd = Path.home() / '.kaggle'; kd.mkdir(exist_ok=True)
    (kd / 'access_token').write_text(token)
    os.makedirs(DATA_DIR, exist_ok=True)
    get_ipython().system(f'kaggle competitions download -c {COMPETITION_SLUG} -p {DATA_DIR}')
    zp = os.path.join(DATA_DIR, f'{COMPETITION_SLUG}.zip')
    if os.path.exists(zp):
        get_ipython().system(f'unzip -q {zp} -d {DATA_DIR}'); os.remove(zp)

train = read_competition_csv(DATA_DIR, 'train')
test = read_competition_csv(DATA_DIR, 'test')
stores = read_competition_csv(DATA_DIR, 'stores')
features = read_competition_csv(DATA_DIR, 'features')
for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])
train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
md = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
train[md] = train[md].fillna(0); test[md] = test[md].fillna(0)
train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)
train = train.sort_values(['Store','Dept','Date']).reset_index(drop=True)
test = test.sort_values(['Store','Dept','Date']).reset_index(drop=True)
log_progress(f'train={train.shape} series={train.groupby(["Store","Dept"]).ngroups}')


In [ ]:
#4 — weekly grid + folds
FAST_RUN = False
N_TRIALS = 5 if FAST_RUN else 25
EPOCHS = 25
MAX_WINDOWS_PER_SERIES = 10
MAX_SERIES = 100 if FAST_RUN else None

FOLDS = [
    {'name': 'holiday', 'start': pd.Timestamp('2011-11-01'), 'end': pd.Timestamp('2012-01-31')},
    {'name': 'spring',  'start': pd.Timestamp('2012-02-01'), 'end': pd.Timestamp('2012-04-30')},
    {'name': 'late',    'start': pd.Timestamp('2012-08-01'), 'end': pd.Timestamp('2012-10-31')},
]

def holiday_weight(s):
    return np.where(pd.Series(s).fillna(0).astype(bool), 5, 1).astype(np.float32)

def build_weekly_grid(df, fill_value=0.0):
    rows = []
    full_idx = pd.date_range(df['Date'].min(), df['Date'].max(), freq='W-FRI')
    for (store, dept), g in df.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').drop_duplicates('Date', keep='last')
        s = g.set_index('Date')['Weekly_Sales'].reindex(full_idx).fillna(fill_value)
        if 'IsHoliday' in g.columns:
            hol = g.set_index('Date')['IsHoliday'].reindex(full_idx).fillna(False).astype(bool)
        else:
            hol = pd.Series(False, index=full_idx)
        rows.append(pd.DataFrame({
            'Store': store, 'Dept': dept, 'Date': full_idx,
            'Weekly_Sales': s.values.astype(np.float64), 'IsHoliday': hol.values,
        }))
    panel = pd.concat(rows, ignore_index=True)
    panel['holiday_weight'] = holiday_weight(panel['IsHoliday'])
    return panel

log_progress('Building weekly grid...')
panel = build_weekly_grid(train)
if MAX_SERIES is not None:
    top = panel.groupby(['Store','Dept']).size().sort_values(ascending=False).head(MAX_SERIES).index
    keep = set(top)
    panel = panel[[(s,d) in keep for s,d in zip(panel['Store'], panel['Dept'])]].copy()
    log_progress(f'top {MAX_SERIES} series')
else:
    log_progress(f'all {panel.groupby(["Store","Dept"]).ngroups} series')

TEST_HORIZON = test['Date'].nunique()
log_progress(f'N_TRIALS={N_TRIALS} EPOCHS={EPOCHS} max_windows/series={MAX_WINDOWS_PER_SERIES} TEST_H={TEST_HORIZON}')
for f in FOLDS:
    n = panel[(panel.Date>=f['start'])&(panel.Date<=f['end'])].Date.nunique()
    log_progress(f"  {f['name']}: {f['start'].date()}→{f['end'].date()} ({n} wks)")


In [ ]:
#5 — N-BEATS + windows (log1p minmax, max 10 windows/series)
class NBEATSBlock(nn.Module):
    def __init__(self, lookback, horizon, hidden, n_layers, dropout=0.0):
        super().__init__()
        layers = []
        for i in range(n_layers):
            layers += [nn.Linear(lookback if i == 0 else hidden, hidden), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.mlp = nn.Sequential(*layers)
        self.backcast = nn.Linear(hidden, lookback)
        self.forecast = nn.Linear(hidden, horizon)
    def forward(self, x):
        h = self.mlp(x)
        return self.backcast(h), self.forecast(h)

class NBEATS(nn.Module):
    def __init__(self, lookback, horizon, n_stacks=2, n_blocks=2, hidden=256, n_layers=4, dropout=0.0):
        super().__init__()
        self.horizon = horizon
        n_total = n_stacks * n_blocks
        self.blocks = nn.ModuleList([
            NBEATSBlock(lookback, horizon, hidden, n_layers, dropout) for _ in range(n_total)
        ])
    def forward(self, x):
        residual, forecast = x, torch.zeros(x.size(0), self.horizon, device=x.device)
        for block in self.blocks:
            backcast, block_f = block(residual)
            residual = residual - backcast
            forecast = forecast + block_f
        return forecast

class WindowDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        s = self.samples[i]
        return (torch.from_numpy(s['x']), torch.from_numpy(s['y_norm']),
                torch.from_numpy(s['y_raw']), torch.from_numpy(s['w']),
                torch.tensor(s['mn'], dtype=torch.float32),
                torch.tensor(s['mx'], dtype=torch.float32))

def fit_minmax_log(sales_log):
    mn, mx = float(np.min(sales_log)), float(np.max(sales_log))
    if mx - mn < 1e-8: mx = mn + 1.0
    return mn, mx

def scale(sales_log, mn, mx):
    return ((sales_log - mn) / (mx - mn + 1e-8)).astype(np.float32)

def inverse_scale(y_norm, mn, mx):
    return np.clip(np.expm1(y_norm * (mx - mn + 1e-8) + mn), 0, None)

def build_windows(panel_df, seq_len, pred_len, train_end, val_start=None, val_end=None,
                  for_train=True, max_per_series=None, seed=42):
    rng = np.random.RandomState(seed)
    samples = []
    for (store, dept), g in panel_df.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').reset_index(drop=True)
        dates = g['Date'].values
        sales = g['Weekly_Sales'].values.astype(np.float64)
        weights = g['holiday_weight'].values.astype(np.float32)
        sales_log = np.log1p(sales)
        train_mask = dates <= np.datetime64(train_end)
        if train_mask.sum() < 2: continue
        mn, mx = fit_minmax_log(sales_log[train_mask])
        norm = scale(sales_log, mn, mx)
        if len(sales) < seq_len + pred_len: continue
        cand = []
        for i in range(len(sales) - seq_len - pred_len + 1):
            t0 = dates[i + seq_len]
            t1 = dates[i + seq_len + pred_len - 1]
            if for_train:
                if t1 > np.datetime64(train_end): continue
            else:
                if t0 < np.datetime64(val_start) or t0 > np.datetime64(val_end): continue
                if t1 > np.datetime64(val_end): continue
            cand.append({
                'x': norm[i:i+seq_len], 'y_norm': norm[i+seq_len:i+seq_len+pred_len],
                'y_raw': sales[i+seq_len:i+seq_len+pred_len].astype(np.float32),
                'w': weights[i+seq_len:i+seq_len+pred_len], 'mn': mn, 'mx': mx,
            })
        if max_per_series is not None and len(cand) > max_per_series:
            idx = rng.choice(len(cand), size=max_per_series, replace=False)
            cand = [cand[j] for j in idx]
        samples.extend(cand)
    return samples

@torch.no_grad()
def eval_wmae(model, loader):
    model.eval(); num = denom = 0.0
    for x, _, y_raw, w, mn, mx in loader:
        pred_n = model(x.to(device)).cpu().numpy()
        pred = inverse_scale(pred_n, mn.numpy().reshape(-1,1), mx.numpy().reshape(-1,1))
        ww = np.nan_to_num(w.numpy(), nan=1.0)
        num += np.sum(ww * np.abs(y_raw.numpy() - pred)); denom += np.sum(ww)
    return float(num / denom) if denom > 0 else float('inf')

def train_one(model, train_loader, val_loader, lr, epochs):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_state, best_wmae = None, float('inf')
    for _ in range(epochs):
        model.train()
        for x, y_norm, *_ in train_loader:
            x, y_norm = x.to(device), y_norm.to(device)
            opt.zero_grad()
            loss = nn.functional.l1_loss(model(x), y_norm)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        wmae = eval_wmae(model, val_loader)
        if wmae < best_wmae:
            best_wmae = wmae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return best_wmae, model

log_progress('N-BEATS helpers ready')


In [ ]:
#6 — Optuna (25 trials × 3 folds)
def suggest_cfg(trial):
    return {
        'seq_len': trial.suggest_int('seq_len', 26, 65, step=13),  # 26,39,52,65
        'pred_len': trial.suggest_categorical('pred_len', [13, 26, 39]),
        'n_stacks': trial.suggest_categorical('n_stacks', [2, 4, 8]),
        'n_blocks': trial.suggest_int('n_blocks', 1, 3),
        'n_layers': trial.suggest_categorical('n_layers', [2, 4]),
        'hidden': trial.suggest_categorical('hidden', [128, 256, 512]),
        'dropout': trial.suggest_categorical('dropout', [0.0, 0.1]),
        'batch_size': trial.suggest_categorical('batch_size', [512, 1024]),
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'epochs': EPOCHS, 'seed': SEED,
    }

def run_fold(cfg, fold):
    train_end = fold['start'] - pd.Timedelta(days=1)
    tr = build_windows(panel, cfg['seq_len'], cfg['pred_len'], train_end, for_train=True,
                       max_per_series=MAX_WINDOWS_PER_SERIES, seed=cfg['seed'])
    va = build_windows(panel, cfg['seq_len'], cfg['pred_len'], train_end,
                       val_start=fold['start'], val_end=fold['end'], for_train=False,
                       max_per_series=None, seed=cfg['seed'])
    if len(tr) < 10 or len(va) < 5:
        return float('inf')
    bs = min(cfg['batch_size'], len(tr))
    tr_loader = DataLoader(WindowDataset(tr), batch_size=bs, shuffle=True, num_workers=0)
    va_loader = DataLoader(WindowDataset(va), batch_size=bs, shuffle=False, num_workers=0)
    torch.manual_seed(cfg['seed'])
    model = NBEATS(cfg['seq_len'], cfg['pred_len'], cfg['n_stacks'], cfg['n_blocks'],
                   cfg['hidden'], cfg['n_layers'], cfg['dropout']).to(device)
    wmae, _ = train_one(model, tr_loader, va_loader, cfg['lr'], cfg['epochs'])
    del model
    if USE_CUDA: torch.cuda.empty_cache()
    return wmae

def objective(trial):
    cfg = suggest_cfg(trial)
    scores = []
    for fold in FOLDS:
        w = run_fold(cfg, fold)
        scores.append(w)
        trial.set_user_attr(f"wmae_{fold['name']}", w)
        if not np.isfinite(w):
            return float('inf')
    mean_w = float(np.mean(scores))
    log_progress(
        f"trial {trial.number}: mean={mean_w:,.2f} | "
        + ' '.join(f"{FOLDS[i]['name']}={scores[i]:,.1f}" for i in range(3))
        + f" | stacks={cfg['n_stacks']} blocks={cfg['n_blocks']} hid={cfg['hidden']} "
        + f"L={cfg['seq_len']} H={cfg['pred_len']} lr={cfg['lr']:.4g}"
    )
    return mean_w

log_progress(f'Optuna start n_trials={N_TRIALS}')
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), study_name='nbeats_v3')
t0 = time.time()
with mlflow.start_run(run_name='NBEATS_Optuna'):
    mlflow.log_params({'notebook_version': NOTEBOOK_VERSION, 'n_trials': N_TRIALS, 'epochs': EPOCHS,
                       'max_windows_per_series': MAX_WINDOWS_PER_SERIES, 'fast_run': FAST_RUN,
                       'preprocess': 'log1p_minmax_weekly_grid', 'folds': 'holiday_spring_late'})
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    mlflow.log_metric('best_mean_wmae', study.best_value)
    mlflow.log_params({f'best_{k}': v for k, v in study.best_params.items()})

log_progress(f'Done { (time.time()-t0)/60:.1f} min | best={study.best_value:,.2f}')
best_cfg = {**study.best_params, 'epochs': EPOCHS, 'seed': SEED}
log_progress(f'best_cfg={best_cfg}')

rows = []
for t in study.trials:
    if t.state.name != 'COMPLETE': continue
    rows.append({'trial': t.number, 'mean_wmae': t.value, **t.params,
                 **{k: t.user_attrs.get(k) for k in ['wmae_holiday','wmae_spring','wmae_late']}})
trials_df = pd.DataFrame(rows).sort_values('mean_wmae')
trials_df.to_csv(STUDY_CSV, index=False)
display(trials_df.head(10))


In [ ]:
#7 — chart + checkpoint
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#8b949e'); ax.title.set_color('white')
vals = [t.value for t in study.trials if t.value is not None and np.isfinite(t.value)]
axes[0].plot(vals, color='#58a6ff')
axes[0].axhline(study.best_value, color='#f78166', ls='--')
axes[0].set_title('Optuna mean WMAE')
top = trials_df.head(8)
axes[1].barh(range(len(top)), top['mean_wmae'], color='#8b949e')
axes[1].barh(0, top.iloc[0]['mean_wmae'], color='#f78166')
axes[1].set_yticks(range(len(top)))
axes[1].set_yticklabels([f't{int(r.trial)}' for r in top.itertuples()], color='#8b949e')
axes[1].invert_yaxis(); axes[1].set_title('Top trials')
plt.tight_layout()
plot_path = WORK_DIR / 'nbeats_optuna.png'
plt.savefig(plot_path, dpi=120, facecolor='#0f1117'); plt.show()

fold = FOLDS[-1]
train_end = fold['start'] - pd.Timedelta(days=1)
tr = build_windows(panel, best_cfg['seq_len'], best_cfg['pred_len'], train_end, for_train=True,
                   max_per_series=MAX_WINDOWS_PER_SERIES)
va = build_windows(panel, best_cfg['seq_len'], best_cfg['pred_len'], train_end,
                   val_start=fold['start'], val_end=fold['end'], for_train=False)
bs = min(best_cfg['batch_size'], max(len(tr), 1))
model = NBEATS(best_cfg['seq_len'], best_cfg['pred_len'], best_cfg['n_stacks'], best_cfg['n_blocks'],
               best_cfg['hidden'], best_cfg['n_layers'], best_cfg['dropout']).to(device)
wmae_late, model = train_one(
    model, DataLoader(WindowDataset(tr), batch_size=bs, shuffle=True),
    DataLoader(WindowDataset(va), batch_size=bs, shuffle=False),
    best_cfg['lr'], best_cfg['epochs'])
torch.save({'state_dict': model.state_dict(), 'config': best_cfg, 'best_mean_wmae': study.best_value,
            'notebook_version': NOTEBOOK_VERSION}, WORK_DIR / 'nbeats_best.pt')
log_progress(f'Saved nbeats_best.pt late_wmae={wmae_late:,.2f}')

with mlflow.start_run(run_name='NBEATS_Best'):
    mlflow.log_params({**best_cfg, 'notebook_version': NOTEBOOK_VERSION})
    mlflow.log_metric('best_mean_fold_wmae', study.best_value)
    mlflow.log_metric('late_fold_wmae', wmae_late)
    mlflow.log_artifact(str(STUDY_CSV)); mlflow.log_artifact(str(plot_path))
    mlflow.log_artifact(str(WORK_DIR / 'nbeats_best.pt'))


In [ ]:
#8 — submission
RUN_SUBMISSION = True
submission_path = None
if RUN_SUBMISSION:
    seq_len = int(best_cfg['seq_len'])
    H = TEST_HORIZON
    train_end = panel['Date'].max()
    samples_H, stats = [], {}
    for (store, dept), g in panel.groupby(['Store', 'Dept']):
        g = g.sort_values('Date').reset_index(drop=True)
        sales = g['Weekly_Sales'].values.astype(np.float64)
        sales_log = np.log1p(sales)
        key = (int(store), int(dept))
        if len(sales) < seq_len + 8: continue
        mn, mx = fit_minmax_log(sales_log); stats[key] = (mn, mx)
        if len(sales) < seq_len + H: continue
        norm = scale(sales_log, mn, mx)
        # subsample windows
        idxs = list(range(len(sales) - seq_len - H + 1))
        if len(idxs) > MAX_WINDOWS_PER_SERIES:
            idxs = list(np.random.RandomState(SEED).choice(idxs, MAX_WINDOWS_PER_SERIES, replace=False))
        for i in idxs:
            samples_H.append({
                'x': norm[i:i+seq_len], 'y_norm': norm[i+seq_len:i+seq_len+H].astype(np.float32),
                'y_raw': sales[i+seq_len:i+seq_len+H].astype(np.float32),
                'w': np.ones(H, np.float32), 'mn': mn, 'mx': mx,
            })
    if len(samples_H) == 0:
        pred_len_sub = int(best_cfg['pred_len'])
        samples_H = build_windows(panel, seq_len, pred_len_sub, train_end, for_train=True,
                                  max_per_series=MAX_WINDOWS_PER_SERIES)
        log_progress(f'fallback pred_len={pred_len_sub}')
    else:
        pred_len_sub = H

    bs = min(int(best_cfg['batch_size']), max(len(samples_H), 1))
    sub_model = NBEATS(seq_len, pred_len_sub, int(best_cfg['n_stacks']), int(best_cfg['n_blocks']),
                       int(best_cfg['hidden']), int(best_cfg['n_layers']),
                       float(best_cfg['dropout'])).to(device)
    loader = DataLoader(WindowDataset(samples_H), batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(sub_model.parameters(), lr=float(best_cfg['lr']), weight_decay=1e-4)
    log_progress(f'Submit retrain windows={len(samples_H)} H={pred_len_sub}')
    for ep in range(1, EPOCHS + 1):
        sub_model.train(); tot=n=0
        for x, y_norm, *_ in loader:
            x, y_norm = x.to(device), y_norm.to(device)
            opt.zero_grad()
            loss = nn.functional.l1_loss(sub_model(x), y_norm)
            loss.backward(); opt.step(); tot += loss.item(); n += 1
        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            log_progress(f'  ep {ep}/{EPOCHS} loss={tot/max(n,1):.4f}')

    pred_map = {}
    sub_model.eval()
    with torch.no_grad():
        for (store, dept), g in panel.groupby(['Store', 'Dept']):
            key = (int(store), int(dept))
            if key not in stats: continue
            sales = g.sort_values('Date')['Weekly_Sales'].values.astype(np.float64)
            if len(sales) < seq_len: continue
            mn, mx = stats[key]
            x = scale(np.log1p(sales[-seq_len:]), mn, mx)
            pred_n = sub_model(torch.tensor(x).unsqueeze(0).to(device)).cpu().numpy()[0]
            yhat = inverse_scale(pred_n, mn, mx)
            if len(yhat) < TEST_HORIZON:
                pattern = sales[-52:] if len(sales) >= 52 else sales
                yhat = np.concatenate([yhat, [pattern[i % len(pattern)] for i in range(len(yhat), TEST_HORIZON)]])
            pred_map[key] = yhat[:TEST_HORIZON]

    rows, n_model, n_naive = [], 0, 0
    all_hist = {(int(s), int(d)): g['Weekly_Sales'].values for (s, d), g in train.groupby(['Store','Dept'])}
    for (store, dept), g in test.groupby(['Store', 'Dept']):
        g = g.sort_values('Date'); key = (int(store), int(dept)); dates = list(g['Date'])
        if key in pred_map:
            yhat = pred_map[key][:len(dates)]; n_model += 1
        else:
            hist = all_hist.get(key, np.array([0.0]))
            pattern = hist[-52:] if len(hist) >= 52 else hist
            yhat = np.array([pattern[i % len(pattern)] for i in range(len(dates))], float); n_naive += 1
        for dt, val in zip(dates, yhat):
            rows.append({'Id': f'{store}_{dept}_{pd.Timestamp(dt).date()}', 'Weekly_Sales': float(max(val, 0))})
    submission = pd.DataFrame(rows)
    submission_path = WORK_DIR / 'submission_nbeats.csv'
    submission.to_csv(submission_path, index=False)
    log_progress(f'Wrote {submission_path} rows={len(submission)} model={n_model} naive={n_naive}')
    display(submission.head())
    with mlflow.start_run(run_name='NBEATS_Submission'):
        mlflow.log_params({**best_cfg, 'submission_horizon': TEST_HORIZON})
        mlflow.log_metric('best_mean_fold_wmae', study.best_value)
        mlflow.log_artifact(str(submission_path))


In [ ]:
#9 — done
summary = {
    'status': 'COMPLETE', 'notebook_version': NOTEBOOK_VERSION,
    'best_mean_fold_wmae': float(study.best_value), 'best_cfg': best_cfg,
    'n_trials': N_TRIALS, 'folds': [f['name'] for f in FOLDS], 'device': str(device),
    'artifacts': [str(STUDY_CSV), str(WORK_DIR/'nbeats_best.pt'), str(WORK_DIR/'nbeats_optuna.png')],
}
if submission_path: summary['artifacts'].append(str(submission_path))
RUN_COMPLETE.write_text(json.dumps(summary, indent=2, default=str))
log_progress('RUN COMPLETE'); print(json.dumps(summary, indent=2, default=str))
